# Decomposition

Using a metabolomics dataset ([MTBLS13411](../../../example_data/MTBLS13411.md)) as an
example, we show how to use the decomposition methods
for the four groups present in the data. The groups are derived based on the sample
names, but you could also merge them from available metadata.

- Principal Component Analysis (PCA)
- Uniform Manifold Approximation and Projection (UMAP)
- t-Distributed Stochastic Neighbor Embedding (t-SNE)

The focus of acore is not plotting, but getting the data in the right format for
plotting. Nonetheless, we show some examples of how to plot the results. VueCore
offers more examples.

In [ ]:
%pip install acore

In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd

import acore.exploratory_analysis as ea
from acore.types.exploratory_analysis import (
    AnnotationResult,
    ThreeComponentSchema,
    ThreeLoadingsSchema,
    TwoComponentSchema,
    TwoLoadingsSchema,
    TwoVariance,
    VarianceRatio,
)

Utility function for plotting

In [ ]:
# replace with vuecore plot
def make_plot(embeddings, annotation: AnnotationResult):
    """Utility function for static plot of dimensionality reductions."""
    map_names = annotation.model_dump(exclude_none=True)
    fig, ax = plt.subplots()
    for i, (group, group_df) in enumerate(embeddings.groupby("group")):
        ax = group_df.rename(columns=map_names).plot.scatter(
            x=annotation.x,
            y=annotation.y,
            label=group,
            c=f"C{i}",
            ax=ax,
        )
    return fig, ax

## Load metabolomics example data

In [ ]:
data = (
    "https://raw.githubusercontent.com/Multiomics-Analytics-Group/acore/"
    "refs/heads/main/"
    "example_data/MTBLS13311/MTBLS13411_processed_data.csv"
)
data = pd.read_csv(data, index_col=0)
# specific to this data, we shorten some column names for better readability
data.columns = data.columns.str.split("(").str[-1].str.replace(")", "")
data

We add the group here based on the sample names. Alternatively you could merge it from
the avilable metadata.

In [ ]:
data["group"] = data.index.str.split("-").str[0]
data["group"].value_counts()

## Principal Component Analysis (PCA)
Show first two principal components of the data.

In [ ]:
results_dfs, annotation = ea.decomposition.run_pca(
    data, drop_cols=[], annotation_cols=[], group="group", components=2, dropna=True
)
pcs, loadings, var_explained = results_dfs

See how much variance is explained by the first two components and validate that
they adhere to the expected format:

In [ ]:
pprint(var_explained)
TwoVariance(pd.Series(var_explained, index=["PC1", "PC2"]))

Show the annotation information for plotting and validate that
they adhere to the expected format:

In [ ]:
pprint(annotation)
annotation = AnnotationResult(**annotation)
map_names = annotation.model_dump(exclude_none=True)
annotation

Show what was computed and validate that
they adhere to the expected format:
- first two principal components of the samples
- loadings for the features on the first two components

We rename the columns for better readability.

In [ ]:
TwoComponentSchema(pcs).rename(columns=map_names).head(10)

The feature communality of the loading is the absolute length of the projection.
So the features listed first here contribute the most to the two first components,
therefore driving the PCA separation.

In [ ]:
TwoLoadingsSchema(loadings).rename(columns=map_names).head(10)

Show the PCA plot of the first two components. VueCore understands the
annotation information (ToDo)

In [ ]:
# replace wit vuecore plot
fig, ax = make_plot(pcs, annotation=annotation)

### More than two components
With more than two components, the results contain an additional `z` column (and
`3`, `4`, ... for further components). The annotation then also has a `z` name.
The `ThreeComponentSchema` and `ThreeLoadingsSchema` validate these results, and
`VarianceRatio` checks that the explained variances are fractions between 0 and 1.

In [ ]:
results_dfs_3, annotation_3 = ea.decomposition.run_pca(
    data, drop_cols=[], annotation_cols=[], group="group", components=3, dropna=True
)
pcs_3, loadings_3, var_explained_3 = results_dfs_3
VarianceRatio(pd.Series(var_explained_3, index=["PC1", "PC2", "PC3"]))
AnnotationResult(**annotation_3)

In [ ]:
ThreeComponentSchema(pcs_3)

In [ ]:
ThreeLoadingsSchema(loadings_3)

## Uniform Manifold Approximation and Projection (UMAP)
Visualize UMAP low-dimensional embedding of the data.
This uses the `umap-learn` package, which is documented with examples at
[umap-learn.readthedocs.io](https://umap-learn.readthedocs.io).

In [ ]:
result, annotation = ea.decomposition.run_umap(
    data,
    drop_cols=["sample", "subject"],
    group="group",
    n_neighbors=10,
    min_dist=0.3,
    metric="cosine",
    dropna=True,
)

In [ ]:
annotation = AnnotationResult(**annotation)
map_names = annotation.model_dump(exclude_none=True)
annotation

In [ ]:
fig, ax = make_plot(result["umap"], annotation=annotation)
TwoComponentSchema(result["umap"]).rename(columns=map_names)

Make sure to check the parameters and tutorials annotations in the API docs at
[umap-learn.readthedocs.io](https://umap-learn.readthedocs.io).

## t-Distributed Stochastic Neighbor Embedding (t-SNE)

Visualize the t-SNE low-dimensional embedding of the data, using
[`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html).
The default `barnes_hut` method can differ slightly between CPU architectures, so we
use the `exact` method and a `random_state` for reproducible results here.

In [ ]:
result, annotation = ea.decomposition.run_tsne(
    data,
    drop_cols=["sample", "subject"],
    group="group",
    perplexity=10,
    method="exact",
    random_state=42,
    dropna=True,
)

In [ ]:
annotation = AnnotationResult(**annotation)
map_names = annotation.model_dump(exclude_none=True)
annotation

In [ ]:
fig, ax = make_plot(result["tsne"], annotation=annotation)
TwoComponentSchema(result["tsne"]).rename(columns=map_names)

Done.